In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

sentence1_counts = df["sentence1"].value_counts()
sentence2_counts = df["sentence2"].value_counts()

df["sentence1_is_repeated"] = df["sentence1"].map(sentence1_counts).gt(1)
df["sentence2_is_repeated"] = df["sentence2"].map(sentence2_counts).gt(1)
df["pair_has_duplicate_text"] = df["sentence1_is_repeated"] | df["sentence2_is_repeated"]
df["pair_is_unique_in_both_columns"] = ~df["pair_has_duplicate_text"]

print({
    "num_examples": len(df),
    "columns": df.columns.tolist(),
    "unique_sentence1": int(df["sentence1"].nunique()),
    "unique_sentence2": int(df["sentence2"].nunique()),
    "pairs_with_duplicate_text": int(df["pair_has_duplicate_text"].sum()),
    "pairs_unique_in_both_columns": int(df["pair_is_unique_in_both_columns"].sum()),
})
print(df.head())


In [ ]:
model = SentenceTransformer(model_name, device=device)

sentence1_texts = df["sentence1"].tolist()
sentence2_texts = df["sentence2"].tolist()

emb1 = model.encode(
    sentence1_texts,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

emb2 = model.encode(
    sentence2_texts,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print({
    "embedding_shape_sentence1": tuple(emb1.shape),
    "embedding_shape_sentence2": tuple(emb2.shape),
    "dtype_sentence1": str(emb1.dtype),
    "dtype_sentence2": str(emb2.dtype),
})


In [ ]:
cosine_scores = np.sum(emb1 * emb2, axis=1).astype(np.float32)
predicted_score_0_5 = ((cosine_scores + 1.0) * 2.5).clip(0.0, 5.0).astype(np.float32)
labels = df["label"].to_numpy(dtype=np.float32)

results_df = df.copy()
results_df["cosine_similarity"] = cosine_scores
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["abs_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

print(results_df[[
    "sentence1", "sentence2", "label", "cosine_similarity",
    "predicted_score_0_5", "abs_error", "sentence1_is_repeated",
    "sentence2_is_repeated", "pair_has_duplicate_text"
]].head(10))


In [ ]:
def compute_metrics(frame):
    y_true = frame["label"].to_numpy(dtype=np.float32)
    y_cos = frame["cosine_similarity"].to_numpy(dtype=np.float32)
    y_pred = frame["predicted_score_0_5"].to_numpy(dtype=np.float32)
    return {
        "num_examples": int(len(frame)),
        "pearson_cosine": float(pearsonr(y_cos, y_true).statistic),
        "spearman_cosine": float(spearmanr(y_cos, y_true).statistic),
        "pearson_predicted_0_5": float(pearsonr(y_pred, y_true).statistic),
        "spearman_predicted_0_5": float(spearmanr(y_pred, y_true).statistic),
        "mae_predicted_0_5": float(np.mean(np.abs(y_pred - y_true))),
    }

overall_metrics = compute_metrics(results_df)
duplicate_metrics = compute_metrics(results_df[results_df["pair_has_duplicate_text"]].copy())
unique_metrics = compute_metrics(results_df[results_df["pair_is_unique_in_both_columns"]].copy())
sentence1_repeated_metrics = compute_metrics(results_df[results_df["sentence1_is_repeated"]].copy())
sentence2_repeated_metrics = compute_metrics(results_df[results_df["sentence2_is_repeated"]].copy())

metrics_table = pd.DataFrame([
    {"subset": "overall", **overall_metrics},
    {"subset": "pair_has_duplicate_text", **duplicate_metrics},
    {"subset": "pair_unique_in_both_columns", **unique_metrics},
    {"subset": "sentence1_repeated", **sentence1_repeated_metrics},
    {"subset": "sentence2_repeated", **sentence2_repeated_metrics},
])

print(metrics_table.to_string(index=False))


In [ ]:
duplicate_error_table = results_df[results_df["pair_has_duplicate_text"]][[
    "sentence1", "sentence2", "label", "cosine_similarity",
    "predicted_score_0_5", "abs_error", "sentence1_is_repeated",
    "sentence2_is_repeated"
]].copy()

unique_error_table = results_df[results_df["pair_is_unique_in_both_columns"]][[
    "sentence1", "sentence2", "label", "cosine_similarity",
    "predicted_score_0_5", "abs_error"
]].copy()

print("worst_duplicate_text_examples")
print(duplicate_error_table.sort_values("abs_error", ascending=False).head(10).to_string(index=False))

print("worst_unique_examples")
print(unique_error_table.sort_values("abs_error", ascending=False).head(10).to_string(index=False))

runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(results_df)}")
print(f"num_pairs_with_duplicate_text: {int(results_df['pair_has_duplicate_text'].sum())}")
print(f"num_pairs_unique_in_both_columns: {int(results_df['pair_is_unique_in_both_columns'].sum())}")
print(f"pearson_cosine_overall: {overall_metrics['pearson_cosine']:.6f}")
print(f"spearman_cosine_overall: {overall_metrics['spearman_cosine']:.6f}")
print(f"pearson_predicted_0_5_overall: {overall_metrics['pearson_predicted_0_5']:.6f}")
print(f"spearman_predicted_0_5_overall: {overall_metrics['spearman_predicted_0_5']:.6f}")
print(f"mae_predicted_0_5_overall: {overall_metrics['mae_predicted_0_5']:.6f}")
print(f"pearson_predicted_0_5_duplicate_text: {duplicate_metrics['pearson_predicted_0_5']:.6f}")
print(f"spearman_predicted_0_5_duplicate_text: {duplicate_metrics['spearman_predicted_0_5']:.6f}")
print(f"mae_predicted_0_5_duplicate_text: {duplicate_metrics['mae_predicted_0_5']:.6f}")
print(f"pearson_predicted_0_5_unique_pairs: {unique_metrics['pearson_predicted_0_5']:.6f}")
print(f"spearman_predicted_0_5_unique_pairs: {unique_metrics['spearman_predicted_0_5']:.6f}")
print(f"mae_predicted_0_5_unique_pairs: {unique_metrics['mae_predicted_0_5']:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
